# Phase 1 — Function Extraction (exploration)

This is a **driver** notebook. The pipeline engine lives in `.py` modules
(`llmcompile/models.py`, `llmcompile/phases/p1_parse.py`); this notebook only
*imports* it and shows the extraction inline. Nothing here holds pipeline state
— re-run top-to-bottom any time and you get the same result.

**Setup:** place this notebook next to the `llmcompile/` folder, i.e.

```
your_workspace/
├── phase1_explore.ipynb   <- this file
└── llmcompile/
    ├── models.py
    └── phases/p1_parse.py
```


In [1]:
# Run once per fresh kernel. Safe to re-run.
%pip install -q llvmlite


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, pathlib
# Make `llmcompile` importable whether the notebook is run from this dir or elsewhere.
here = pathlib.Path.cwd()
if (here / "llmcompile").exists() and str(here) not in sys.path:
    sys.path.insert(0, str(here))

from llmcompile.phases.p1_parse import parse_module, summarize
from llmcompile.models import Verdict
print("imports OK")

imports OK


## 1. A sample `-O0`-style module

A small but representative module: a global, a foreign `declare`, and two
definitions where `@use` calls both its sibling `@add` and the external
`@external`. Swap this for your own `clang -O0 -emit-llvm -S` output when ready.


In [3]:
SAMPLE_IR = r"""
; ModuleID = 'sample'
source_filename = "sample.c"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

@counter = global i32 0, align 4

declare i32 @external(i32)

define i32 @add(i32 %a, i32 %b) {
entry:
  %r = add i32 %a, %b
  ret i32 %r
}

define i32 @use(i32 %x) {
entry:
  %t = call i32 @add(i32 %x, i32 1)
  %e = call i32 @external(i32 %t)
  ret i32 %e
}
"""

# To use a real file instead:
# SAMPLE_IR = open("yourfile.ll").read()

## 2. Parse and summarise

`parse_module` parses with llvmlite, runs `verify()`, then extracts one record
per **definition** (the `declare` is shared context, not a record).


In [4]:
parsed = parse_module(SAMPLE_IR)
print(summarize(parsed))
print()
print("records:", [r.name for r in parsed.functions])
print("all PENDING (Phase 5 not run):",
      all(r.verdict is Verdict.PENDING for r in parsed.functions))

Extracted 2 function definition(s):
  - add                      (16 lines of standalone IR)
  - use                      (17 lines of standalone IR)

records: ['add', 'use']
all PENDING (Phase 5 not run): True


## 3. The extracted standalone IR, per function

Each record's `original_ir` is self-contained: shared preamble + one-line
`declare`s for sibling functions + this function's full body.


In [5]:
for rec in parsed.functions:
    print("=" * 70)
    print(f"FUNCTION: {rec.name}")
    print("=" * 70)
    print(rec.original_ir)

FUNCTION: add
; ModuleID = '<string>'
source_filename = "sample.c"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

@counter = global i32 0, align 4

declare i32 @external(i32)
; --- declarations of sibling functions ---
declare i32 @use(i32 %x)

define i32 @add(i32 %a, i32 %b) {
entry:
  %r = add i32 %a, %b
  ret i32 %r
}

FUNCTION: use
; ModuleID = '<string>'
source_filename = "sample.c"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

@counter = global i32 0, align 4

declare i32 @external(i32)
; --- declarations of sibling functions ---
declare i32 @add(i32 %a, i32 %b)

define i32 @use(i32 %x) {
entry:
  %t = call i32 @add(i32 %x, i32 1)
  %e = call i32 @external(i32 %t)
  ret i32 %e
}



## 4. Sibling calls become declarations, not inlined bodies

`@use` calls `@add`, so its standalone IR carries a **declaration** of `@add`
(references resolve) but **not** `@add`'s body (no token bloat, nothing for the
LLM to mangle). The leaf `@add` carries its own body in full.


In [6]:
use = next(r for r in parsed.functions if r.name == "use")
add = next(r for r in parsed.functions if r.name == "add")

print("@use declares @add        :", "declare" in use.original_ir and "@add" in use.original_ir)
print("@use omits @add's body    :", "%r = add i32 %a, %b" not in use.original_ir)
print("@use keeps @external decl :", "@external" in use.original_ir)
print("@use keeps @counter global:", "@counter" in use.original_ir)
print("@add carries its own body :", "%r = add i32 %a, %b" in add.original_ir)

@use declares @add        : True
@use omits @add's body    : True
@use keeps @external decl : True
@use keeps @counter global: True
@add carries its own body : True


## 5. Proof: each function is independently assemblable

The load-bearing property for Phase 5 — every extracted snippet must re-parse
and re-verify on its own. If this cell runs clean, the standalone extraction is
sound.


In [7]:
import llvmlite.binding as llvm
llvm.initialize()

for rec in parsed.functions:
    m = llvm.parse_assembly(rec.original_ir)   # raises if the IR is broken
    m.verify()
    defs = {fn.name for fn in m.functions if not fn.is_declaration}
    assert rec.name in defs, f"{rec.name} missing after re-parse"
    print(f"{rec.name:<10} re-parses + verifies  ✓   (definitions present: {sorted(defs)})")

add        re-parses + verifies  ✓   (definitions present: ['add'])
use        re-parses + verifies  ✓   (definitions present: ['use'])


## 6. The retained module (for Phase 6 fallback)

`parsed.module_ref` is the live llvmlite module kept in memory for the whole
run; `parsed.preamble` is the shared module-level context. Phase 6 will reach
back into these to reinsert any function whose optimisation fails verification.


In [8]:
print("module_ref retained:", parsed.module_ref is not None)
print()
print("---- shared preamble ----")
print(parsed.preamble)

module_ref retained: True

---- shared preamble ----
; ModuleID = '<string>'
source_filename = "sample.c"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

@counter = global i32 0, align 4

declare i32 @external(i32)


## Where notebooks fit from here

This is the right division of labour for the project's reproducibility story:

* **Engine** stays in `.py` — `models.py`, `phases/`, `verification/`, `orchestrator.py`.
* **Notebooks** import it for exploration, experiments, and dissertation
  figures: e.g. visualising instruction count before/after a transform, or
  tabulating Phase 5 verdicts across a corpus.

Next up: **Phase 2** — cyclomatic complexity + token count over
`parsed.functions`, feeding the triage threshold and the Phase 3 routing tiers.
